# DJAI — Clean local development notebook

**Stage 1: playlist → audio features → segments → transition candidates.** Run top-to-bottom in VS Code with the project virtual environment.

This replaces Colab-specific ingestion and removes visualization experiments, duplicate exports, and exploratory calculations. The original scoring formulas are retained as the **rule-based baseline**. The VAE, learned arranger, and renderer are **not implemented yet**.

**Important:** The attached source notebook ends at the heading “Testing what we have so far with real audio”; it does not contain the working audio-rendering engine discussed earlier. Preserve that engine separately if it lives in another notebook/file.

## 1 — Imports and local paths
Place WAV/MP3/FLAC files in `data/audio/` (not tracked by Git).

In [ ]:
from pathlib import Path
from dataclasses import dataclass
import numpy as np
import pandas as pd
import librosa

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
AUDIO_DIR = PROJECT_ROOT / "data" / "audio"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_EXTENSIONS = {".wav", ".mp3", ".flac", ".aiff", ".ogg"}
audio_paths = sorted(p for p in AUDIO_DIR.rglob("*") if p.suffix.lower() in AUDIO_EXTENSIONS)
print(f"Found {len(audio_paths)} audio files in {AUDIO_DIR}")
if not audio_paths:
    print("Add audio files to data/audio/ before running the remaining cells.")

## 2 — Audio analysis and segment extraction
We use **non-overlapping 16-beat windows (4 bars in 4/4)** as an initial baseline. This is *not* reliable downbeat/phrase identification. Key is estimated at track level; energy is measured per segment. Beat times stored in `Segment` are **segment-relative**.

In [ ]:
def estimate_key(y, sr):
    # Compute chromagram
    chroma = librosa.feature.chroma_cqt(y=y, sr=sr)
    chroma_vals = np.sum(chroma, axis=1)

    # Major and minor pitch profiles
    maj_profile = np.array([6.35, 2.23, 3.48, 2.33, 4.38, 4.09, 2.52, 5.19, 2.39, 3.66, 2.29, 2.88])
    min_profile = np.array([6.33, 2.68, 3.52, 5.38, 2.60, 3.53, 2.54, 4.75, 3.98, 2.69, 3.34, 3.17])

    keys = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']

    maj_corrs = [np.corrcoef(chroma_vals, np.roll(maj_profile, i))[0, 1] for i in range(12)]
    min_corrs = [np.corrcoef(chroma_vals, np.roll(min_profile, i))[0, 1] for i in range(12)]

    if max(maj_corrs) > max(min_corrs):
        return f"{keys[np.argmax(maj_corrs)]} Major"
    else:
        return f"{keys[np.argmax(min_corrs)]} Minor"

In [ ]:
@dataclass
class Segment:
    segment_id: str
    song_id: str
    audio_path: str
    start_time: float
    end_time: float
    bpm: float
    beat_times: np.ndarray  # seconds relative to segment start
    key: str
    mode: str
    energy: float
    phrase_length_bars: float
    exit_boundary_confidence: float = 0.5  # unverified boundaries
    entry_boundary_confidence: float = 0.5


def extract_segments(path, beats_per_segment=16):
    y, sr = librosa.load(path, sr=22050, mono=True)
    if y.size == 0:
        return []
    tempo, beat_frames = librosa.beat.beat_track(y=y, sr=sr)
    bpm = float(np.asarray(tempo).reshape(-1)[0])
    beat_times = librosa.frames_to_time(beat_frames, sr=sr)
    if bpm <= 0 or len(beat_times) < beats_per_segment + 1:
        return []
    key_label = estimate_key(y, sr)
    key, mode = key_label.split(" ", 1)
    result = []
    for first in range(0, len(beat_times) - beats_per_segment, beats_per_segment):
        start = float(beat_times[first]); end = float(beat_times[first + beats_per_segment])
        clip = y[int(start * sr):int(end * sr)]
        if clip.size == 0:
            continue
        relative_beats = beat_times[first:first + beats_per_segment + 1] - start
        result.append(Segment(
            segment_id=f"{path.stem}:{first}", song_id=path.stem,
            audio_path=str(path.resolve()), start_time=start, end_time=end,
            bpm=bpm, beat_times=relative_beats,
            key=key, mode=mode, energy=float(np.sqrt(np.mean(clip**2))),
            phrase_length_bars=beats_per_segment / 4,
        ))
    return result

segments = []
for path in audio_paths:
    try:
        found = extract_segments(path)
        segments.extend(found)
        print(f"{path.name}: {len(found)} segments")
    except Exception as exc:
        print(f"SKIPPED {path.name}: {exc}")
print(f"Total segments: {len(segments)}")

## 3 — Rule-based compatibility baseline (preserved from source notebook)
These functions are retained for comparison with a future learned scorer. Scores are **heuristics**, not validated judgments of musical quality.

In [ ]:
# Initial sigma value (sensitivity parameter) = 0.10

def calculate_tempo_compatibility(bpm_a, bpm_b, sigma=0.10):

    if sigma <= 0: #make sure that we have initialized our sigma value!
        raise ValueError("Sigma must be positive.")

    #these are the k values from above, essentially it's our allowed speed ratios
    ratios = [0.5, 1.0, 2.0]

    #relative tempo differences
    differences = []

    for k in ratios:
        #and then we use the log to measure proportional differences (not absolute)
        difference = abs(np.log(bpm_a / (k * bpm_b)))
        differences.append((difference, k))

    #we want to minimize the difference (as we see in our function above)
    #we are essentially testing all speeds and selecting the most compatible one.
    relative_difference, best_k = min(differences, key=lambda x: x[0])

    #transform the distance into a score between 0 and 1
    compatibility = np.exp(
        -(relative_difference ** 2) / (2 * sigma ** 2)
    )

    return compatibility, best_k

In [ ]:
##now we actually calculate this
##we USE A CIRCULAR MEAN because beat phases repeat every beat period (avoids issues when phase values wrap around the end of a beat interval)
def calculate_phase_offset(beats_a, beats_b, bpm):

    # Convert beat timestamps to arrays
    beats_a = np.asarray(beats_a)
    beats_b = np.asarray(beats_b)

    #beat period calculation
    beat_period = 60 / bpm

    #convert timestamps into circular phase angles
    phase_a = 2 * np.pi * beats_a / beat_period
    phase_b = 2 * np.pi * beats_b / beat_period

    #calculate average circular phase
    mean_a = np.angle(np.mean(np.exp(1j * phase_a)))
    mean_b = np.angle(np.mean(np.exp(1j * phase_b)))

    #calculate phase difference
    phase_difference = np.angle(np.exp(1j * (mean_b - mean_a)))

    #convert phase difference back into seconds
    time_offset = phase_difference * beat_period / (2 * np.pi)

    return time_offset

In [ ]:
def calculate_alignment_quality(beats_a, beats_b, bpm, sigma=0.03):
  #sigma controls how sensitve the function is to timing inconsistencies.
  # 0.03 means we're using approximately 30 millisenconds as our timing-error scale
  #this parameter will eventually be tuned by our model...
    beats_a = np.asarray(beats_a, dtype=float)
    beats_b = np.asarray(beats_b, dtype=float)

    beat_period = 60 / bpm

    phase_a = 2 * np.pi * beats_a / beat_period
    phase_b = 2 * np.pi * beats_b / beat_period

    concentration_a = abs(np.mean(np.exp(1j * phase_a)))
    concentration_b = abs(np.mean(np.exp(1j * phase_b)))

    # Convert circular concentration into a timing-error estimate.
    # This is an approximate measure, not an exact RMS calculation.
    concentration = np.sqrt(
        concentration_a * concentration_b
    )

    timing_error = np.sqrt(
        max(0, -2 * np.log(max(concentration, 1e-12)))
    ) * beat_period / (2 * np.pi)

    alignment_score = np.exp(
        -(timing_error ** 2) / (2 * sigma ** 2)
    )

    return alignment_score

In [ ]:
def calculate_rhythm_compatibility(
    bpm_a,
    bpm_b,
    beats_a,
    beats_b
): #we need beats and bpm for both input segments
#we can calculate the bpm difference without adjusting anything
    tempo_score, multiplier = calculate_tempo_compatibility(
        bpm_a,
        bpm_b
    )

    #adjust Segment B's beat timestamps to its matched playback speed.
    #we are essentially shifting the beats of B like we talked about earlier
    adjusted_beats_b = np.asarray(beats_b, dtype=float) / multiplier

    #the alignment score measures the consistency of the detected grids
    alignment_score = calculate_alignment_quality(
        beats_a,
        adjusted_beats_b, #we use the newly aligned beats of segment B here
        bpm_a
    )
    #the phase offset tells the arranger how to position B
    phase_offset = calculate_phase_offset(
        beats_a,
        adjusted_beats_b,
        bpm_a
    )

    rhythm_score = tempo_score * alignment_score #the overall rhythm score is our two calculated scores combined

    return {
        "rhythm_score": rhythm_score,
        "tempo_score": tempo_score,
        "alignment_score": alignment_score,
        "tempo_multiplier": multiplier,
        "phase_offset": phase_offset
    }

In [ ]:
#create a dictionary assigning each pitch class a postion
circle_of_fifths = {
    "C": 0,
    "G": 1,
    "D": 2,
    "A": 3,
    "E": 4,
    "B": 5,
    "F#": 6,
    "C#": 7,
    "G#": 8,
    "D#": 9,
    "A#": 10,
    "F": 11
}

#there's also some names that represent the same pitch (I should've studied music theory)
enharmonic_equivalents = {
    "Db": "C#",
    "Eb": "D#",
    "Gb": "F#",
    "Ab": "G#",
    "Bb": "A#"
}
#there are also some major+minor relationships that share the same notes. IDK WHAT THIS MEANS FULLY THIS IS DEEP MUSIC THEORY
#to whoever is reading this, I wouldn't worry about it much, just know that these relationships help assign a higher compatibility score
pitch_classes = {
    "C": 0,
    "C#": 1,
    "D": 2,
    "D#": 3,
    "E": 4,
    "F": 5,
    "F#": 6,
    "G": 7,
    "G#": 8,
    "A": 9,
    "A#": 10,
    "B": 11
}

In [ ]:
#the distance between two keys is the shortest number of steps between them on the circle
#since the circle is, well, a circle, we use d=min(|i-j|, 12-|i-j|)

def calculate_key_distance(key_a, key_b):

    key_a = enharmonic_equivalents.get(key_a, key_a)
    key_b = enharmonic_equivalents.get(key_b, key_b)

    if key_a not in circle_of_fifths:
        raise ValueError(f"Unknown key: {key_a}")

    if key_b not in circle_of_fifths:
        raise ValueError(f"Unknown key: {key_b}")

    position_a = circle_of_fifths[key_a]
    position_b = circle_of_fifths[key_b]

    distance = abs(position_a - position_b)

    return min(distance, 12 - distance)

In [ ]:
def calculate_harmonic_compatibility(key_a, mode_a, key_b, mode_b):

#one thing that is important to note but not memorize is we use a custom score system
#THESE ARE JUST INITIAL WEIGHTS! We will tune them later within the model

# Same key = 1.00
# Relative major/minor = 0.95
# One step apart = 0.85
# Two steps apart = 0.65
# Three steps apart = 0.45
# Four or more steps apart = 0.25

#you'll see these scores below

    key_a = enharmonic_equivalents.get(key_a, key_a)
    key_b = enharmonic_equivalents.get(key_b, key_b)

    mode_a = mode_a.lower()
    mode_b = mode_b.lower()

    # Same tonic and mode
    if key_a == key_b and mode_a == mode_b:
        return 1.0 #aka same key

    # Check relative major/minor relationship
    semitone_difference = (
        pitch_classes[key_b] - pitch_classes[key_a]
    ) % 12

    if mode_a == "major" and mode_b == "minor":
        if semitone_difference == 9:
            return 0.95 #aka relative major/minor

    if mode_a == "minor" and mode_b == "major":
        if semitone_difference == 3:
            return 0.95 #aka relative major/minor

    # Same tonic, different mode
    if key_a == key_b:
        return 0.65 #aka two steps apart

    # Circle-of-fifths relationship
    distance = calculate_key_distance(key_a, key_b)

    scores = {
        1: 0.85, #if the relationship is one step apart on the circle
        2: 0.65, #if the relationship is two steps apart
        3: 0.45 #if three steps apart
    }

    if distance == 0:
        return 0.65

    return scores.get(distance, 0.25)

In [ ]:
def calculate_energy_compatibility(
    energy_a,
    energy_b,
    target_delta=0.0,
    sigma=0.25
):

    energy_difference = energy_b - energy_a #this is the energy difference discussed above

    #and this is our Gaussian. Basically,
    #it is e to the power of the negative of actual energy difference minus target energy diference square
    #over 2 times the tolerance value squared
    compatibility = np.exp(
        -((energy_difference - target_delta) ** 2)
        / (2 * sigma ** 2)
    )

    return compatibility

In [ ]:
def calculate_phrase_length_compatibility(
    phrase_a,
    phrase_b,
    sigma=0.15
):

    ratios = [0.5, 1.0, 2.0] #we allow 0.5, 1, and 2x phrase length relationships (similar to tempo system)
                              #so 4bars and 8bars is more compatible than we'd assume because we allow a 2x phrase length!

    differences = []

    for ratio in ratios:

        difference = abs(
            np.log(phrase_a / (ratio * phrase_b))
        )

        differences.append(difference)

    best_difference = min(differences) #we want to reward compatible phrase lengths as in 8bars and 8bars is excelletn
                                        #whereas 8bars and 7bars is less compatible

    compatibility = np.exp(
        -(best_difference ** 2) / (2 * sigma ** 2) #this is our gaussian function
    )

    return compatibility

In [ ]:
def calculate_boundary_compatibility(
    exit_confidence, #how confident are we that segment a is approaching a suitable ending
    entry_confidence #how confident are we that segment b is approaching a suitable ending
):

    return np.sqrt(
        exit_confidence * entry_confidence #we use the mean because we want both boundaries to be strong!
    )

In [ ]:
#we have both phrase length and boundary compatibility calculations... so we combine them
def calculate_phrase_compatibility(
    phrase_a,
    phrase_b,
    exit_confidence,
    entry_confidence
):

    length_score = calculate_phrase_length_compatibility(
        phrase_a,
        phrase_b
    )

    boundary_score = calculate_boundary_compatibility(
        exit_confidence,
        entry_confidence
    )

    phrase_score = length_score * boundary_score

    return {
        "phrase_score": phrase_score,
        "length_score": length_score,
        "boundary_score": boundary_score
    }

In [ ]:
def evaluate_transition(segment_a, segment_b):
#our method will receieve two segments, calculate rhythm/harmonic/energy/phrase compatibility and combine the scores using our weights.
#then return the overall score and all component scores


    # -----------------------------
    # 1. RHYTHM COMPATIBILITY
    # -----------------------------

    rhythm_results = calculate_rhythm_compatibility(
        segment_a.bpm,
        segment_b.bpm,
        segment_a.beat_times,
        segment_b.beat_times
    )

    rhythm_score = rhythm_results["rhythm_score"]


    # -----------------------------
    # 2. HARMONIC COMPATIBILITY
    # -----------------------------

    harmonic_score = calculate_harmonic_compatibility(
        segment_a.key,
        segment_a.mode,
        segment_b.key,
        segment_b.mode
    )


    # -----------------------------
    # 3. ENERGY COMPATIBILITY
    # -----------------------------

    energy_score = calculate_energy_compatibility(
        segment_a.energy,
        segment_b.energy
    )


    # -----------------------------
    # 4. PHRASE COMPATIBILITY
    # -----------------------------

    phrase_results = calculate_phrase_compatibility(
        segment_a.phrase_length_bars,
        segment_b.phrase_length_bars,
        segment_a.exit_boundary_confidence,
        segment_b.entry_boundary_confidence
    )

    phrase_score = phrase_results["phrase_score"]


    # -----------------------------
    # 5. COMBINE SCORES
    # -----------------------------

    weights = {
        "rhythm": 0.35,
        "harmonic": 0.25,
        "energy": 0.20,
        "phrase": 0.20
    }

    transition_score = (
        rhythm_score * weights["rhythm"] +
        harmonic_score * weights["harmonic"] +
        energy_score * weights["energy"] +
        phrase_score * weights["phrase"]
    )


    # -----------------------------
    # 6. RETURN RESULTS
    # -----------------------------

    return {
        "transition_score": transition_score,

        "rhythm_score": rhythm_score,

        "harmonic_score": harmonic_score,

        "energy_score": energy_score,

        "phrase_score": phrase_score,

        "tempo_multiplier": rhythm_results["tempo_multiplier"],

        "phase_offset": rhythm_results["phase_offset"],

        "rhythm_details": rhythm_results,

        "phrase_details": phrase_results
    }

## 4 — Automatic candidate generation
Exclude self-transitions and same-song transitions for this initial demonstration. Limit candidates per segment to avoid quadratic memory growth for large playlists.

In [ ]:
def generate_transition_candidates(segments, top_k=5):
    rows = []
    for a in segments:
        choices = []
        for b in segments:
            if a.song_id == b.song_id:
                continue
            scores = evaluate_transition(a, b)
            choices.append({
                "from_segment_id": a.segment_id,
                "to_segment_id": b.segment_id,
                "score": float(scores["transition_score"]),
                "tempo_multiplier": float(scores["tempo_multiplier"]),
                "phase_offset": float(scores["phase_offset"]),
            })
        rows.extend(sorted(choices, key=lambda r: r["score"], reverse=True)[:top_k])
    return pd.DataFrame(rows, columns=[
        "from_segment_id", "to_segment_id", "score", "tempo_multiplier", "phase_offset"
    ])

candidates_df = generate_transition_candidates(segments)
print(f"Candidate transitions: {len(candidates_df)}")
display(candidates_df.head(10))
if not candidates_df.empty:
    candidates_df.to_csv(OUTPUT_DIR / "transition_candidates.csv", index=False)

## 5 — Next implementation slots (do not paste code randomly)

1. **ArrangementPlan + TransitionInstruction:** insert after the candidate generation cell. Define the full ordered plan and transition parameters.
2. **Baseline planner:** select a coherent sequence from the candidate table, with reuse/energy constraints.
3. **VAE:** add feature matrix preparation, normalization, training, and embedding extraction in a separate section *after* the baseline is working.
4. **Learned transition scorer:** compare against `evaluate_transition` rather than deleting it.
5. **Audio renderer:** integrate the separate working engine after the planner is producing instructions.

**Known limitations:** No confirmed downbeats; fixed-length windows may cut phrases; track-level key estimates are approximate; raw RMS is not normalized across tracks; current transition scores are not trained; no audio is rendered here.